# Clase 5 — Introducción a modelos de visión

## Pregunta central

> **¿Qué tipo de respuesta puede dar una IA cuando recibe una imagen?**

En esta clase no vamos a entrenar redes neuronales ni a estudiar su matemática interna.
El objetivo es **ver modelos de visión funcionando** y entender qué entra, qué sale
y cómo interpretar el resultado.

Trabajaremos con tres tareas:

| Tarea | Pregunta simple | Respuesta del modelo |
|---|---|---|
| Clasificación | ¿Qué hay principalmente en la imagen? | Una o varias categorías |
| Detección | ¿Qué objetos hay y dónde están? | Categorías + rectángulos |
| Segmentación | ¿Qué píxeles pertenecen a una región? | Una máscara |

## Objetivos

Al finalizar deberías poder:

- distinguir clasificación, detección y segmentación;
- entender que una imagen debe prepararse antes de entrar al modelo;
- ejecutar un modelo preentrenado de clasificación y otro de detección;
- interpretar un `score` como una señal de confianza, no como una verdad;
- comprender de forma visual qué significa IoU;
- experimentar cambiando una entrada o un umbral y explicar qué ocurrió.

## Cómo trabajar con este notebook

1. Ejecutá las celdas en orden.
2. Mirá primero el resultado.
3. Después leé la explicación.
4. Modificá únicamente las variables indicadas en las actividades.
5. No hace falta entender cada línea de PyTorch o YOLO para cumplir el objetivo.

La pregunta que debe acompañar toda la clase es:

> **¿Qué información recibe el modelo y qué información devuelve?**

---
## 1. Tres maneras de analizar una imagen

Supongamos que tenemos una fotografía con un perro.

Un sistema de visión puede responder preguntas distintas:

```text
CLASIFICACIÓN
imagen → "perro"

DETECCIÓN
imagen → "perro" + dónde está

SEGMENTACIÓN
imagen → qué píxeles forman el perro
```

### Clasificación

El modelo observa la imagen completa y propone una categoría.

### Detección

Además de reconocer el objeto, indica su posición dibujando una **caja**
o rectángulo alrededor. Una imagen puede tener ninguna, una o muchas detecciones.

### Segmentación

En lugar de aproximar la posición con un rectángulo, intenta marcar la región
del objeto píxel por píxel.

### La idea importante

No existe una tarea mejor en todos los casos. La elección depende de la pregunta.

- Si solo queremos saber si una foto contiene un auto, clasificación puede alcanzar.
- Si queremos contar autos y saber dónde están, necesitamos detección.
- Si necesitamos conocer el contorno o el área exacta, necesitamos segmentación.

En todos los casos ocurre lo mismo:

```text
imagen → modelo → resultado → decisión humana o de otro sistema
```

## Glosario mínimo

No es necesario memorizar estos términos. Se incluyen para reconocerlos cuando
aparecen en el código.

| Término | Significado en esta clase |
|---|---|
| Píxel | Un punto de la imagen |
| Canal RGB | Información de rojo, verde y azul |
| Tensor | La imagen convertida en números para que el modelo pueda procesarla |
| Modelo preentrenado | Modelo que ya aprendió usando muchas imágenes |
| Inferencia | Usar un modelo ya entrenado para obtener una predicción |
| Clase / label | Nombre de una categoría |
| Score | Valor que acompaña una predicción; cuanto mayor, más fuerte fue la respuesta |
| Caja / bounding box | Rectángulo que ubica un objeto |
| Máscara | Región marcada píxel por píxel |
| Threshold / umbral | Valor mínimo que exigimos para aceptar o mostrar un resultado |
| Ground truth | Respuesta correcta usada como referencia para evaluar |

---
## 2. ¿Cómo entra una imagen a un modelo?

Para nosotros una fotografía es una imagen. Para una computadora es una gran
tabla de números.

En una imagen RGB cada píxel guarda tres valores: rojo, verde y azul.

Los modelos reciben esos números en una estructura llamada **tensor**.
No necesitamos estudiar tensores en profundidad. Para esta clase alcanza con
recordar este flujo:

```text
archivo JPG/PNG
      ↓
se abre la imagen
      ↓
se ajusta al formato esperado
      ↓
se convierte en números
      ↓
modelo
```

### ¿Qué hace una CNN?

Modelos clásicos como ResNet pertenecen a una familia llamada **CNN**.

La intuición es sencilla: la red aprende patrones visuales útiles.

```text
píxeles
  ↓
bordes y contrastes
  ↓
texturas y formas
  ↓
combinaciones más complejas
  ↓
predicción
```

Durante el **entrenamiento**, el modelo aprende qué patrones son útiles.
Durante la **inferencia**, aplica lo aprendido a una nueva imagen.

En este notebook hacemos inferencia: usaremos modelos que ya vienen entrenados.

### Preparar una imagen y crear variaciones

Antes de enviar una imagen a un modelo suele ser necesario prepararla.

El **preprocesamiento** adapta la imagen al formato que el modelo espera.
Por ejemplo:

- cambiar el tamaño;
- convertirla a RGB;
- convertir sus píxeles a números;
- ajustar la escala de esos números.

La regla práctica es simple:

> Si usamos un modelo preentrenado, conviene usar también el preprocesamiento
> recomendado para ese modelo.

### Data augmentation

Durante el entrenamiento también podemos crear versiones modificadas de una
imagen: girarla un poco, cambiar el brillo o hacer un espejo horizontal.

A esto se lo llama **data augmentation**. Su objetivo es que el modelo vea
más variedad.

Pero una transformación solo sirve si **no cambia la respuesta correcta**.

Primero vamos a verlo de forma visual.

In [ ]:
# ------------------------------------------------------------
# PREPARACIÓN DEL NOTEBOOK
# ------------------------------------------------------------
# Esta celda importa las librerías y carga imágenes de ejemplo.
# No hace falta memorizar las importaciones.
#
# Al terminar tendremos un diccionario llamado `imagenes`
# con las fotos que usaremos en los experimentos.

from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.datasets import load_sample_image
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import functional as TF

# Modo rápido para que los ejemplos sean razonables en una notebook común.
FAST_MODE = True

# Semilla fija: ayuda a repetir resultados cuando hay azar.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))


def encontrar_imagen(nombre):
    """Busca una imagen dentro de las carpetas habituales del curso."""
    candidatos = [
        Path.cwd() / "assets" / "images" / nombre,
        Path.cwd() / "IA para programadores" / "assets" / "images" / nombre,
    ]
    for padre in Path.cwd().parents:
        candidatos.append(
            padre / "IA para programadores" / "assets" / "images" / nombre
        )
    return next(
        (ruta.resolve() for ruta in candidatos if ruta.exists()),
        None,
    )


# Intentamos usar primero las imágenes propias del curso.
ruta_china = encontrar_imagen("china.jpg")
ruta_flor = encontrar_imagen("flower.jpg")
ruta_deteccion = encontrar_imagen("escena_bus_peatones.jpg")

# Si faltan las dos primeras, scikit-learn aporta imágenes de muestra.
imagenes = {
    "paisaje": (
        Image.open(ruta_china).convert("RGB")
        if ruta_china
        else Image.fromarray(load_sample_image("china.jpg")).convert("RGB")
    ),
    "flor": (
        Image.open(ruta_flor).convert("RGB")
        if ruta_flor
        else Image.fromarray(load_sample_image("flower.jpg")).convert("RGB")
    ),
}

if ruta_deteccion:
    imagenes["deteccion_calle"] = Image.open(ruta_deteccion).convert("RGB")

print("Modo rápido:", FAST_MODE)
print("PyTorch:", torch.__version__)
print("Imágenes disponibles:", list(imagenes))

## Experimento A — una misma imagen, distintas versiones

Vamos a tomar una imagen y generar varias versiones.

No estamos intentando mejorarla. Queremos observar qué modificaciones podrían
aparecer durante el entrenamiento de un modelo.

Antes de ejecutar la celda, pensá:

> **¿Una flor deja de ser una flor si la giro 20° o cambio su brillo?**

Después compará las variantes.

In [ ]:
# Elegimos una imagen base.
imagen_base = imagenes["flor"]

# Creamos varias versiones de ESA MISMA imagen.
variantes = {
    "original": imagen_base,
    "espejo horizontal": TF.hflip(imagen_base),
    "rotación +20°": TF.rotate(imagen_base, angle=20),
    "menos brillo": TF.adjust_brightness(imagen_base, 0.55),
    "más contraste": TF.adjust_contrast(imagen_base, 1.7),
}

# Las mostramos juntas para comparar visualmente.
fig, axes = plt.subplots(1, len(variantes), figsize=(17, 4))

for ax, (nombre, variante) in zip(axes, variantes.items()):
    ax.imshow(variante)
    ax.set_title(nombre)
    ax.axis("off")

plt.suptitle("Una imagen, varias transformaciones", y=1.03, fontsize=14)
plt.tight_layout()
plt.show()

### Para observar

1. ¿Seguís viendo el mismo objeto en todas?
2. ¿Cuál de las transformaciones cambia más la apariencia?
3. ¿Se te ocurre un caso donde hacer espejo horizontal sería peligroso?
4. ¿Por qué sería útil mostrar distintas versiones de una imagen a un modelo durante su entrenamiento?

La idea del experimento es comprender que **más variedad visual puede ayudar**,
siempre que la respuesta correcta siga siendo la misma.

---
## 3. Modelos preentrenados

Entrenar un modelo desde cero puede requerir muchísimas imágenes, tiempo y cómputo.

Por eso es común comenzar con un **modelo preentrenado**: un modelo que ya
aprendió patrones visuales usando una colección grande de imágenes.

```text
muchas imágenes
      ↓
entrenamiento previo
      ↓
modelo que ya reconoce patrones visuales
      ↓
lo usamos con una nueva imagen
```

También se puede adaptar después a un problema propio. Ese proceso se conoce
como **transfer learning**.

En esta introducción no vamos a entrenar ni adaptar modelos. Solo vamos a
usarlos para observar cómo responden.

### Una limitación importante

Un modelo preentrenado no conoce automáticamente nuestras categorías particulares.
Puede ser un excelente punto de partida, pero siempre hay que probarlo con datos
del problema real.

### Algunos nombres que vas a encontrar

No necesitamos estudiar estas arquitecturas en detalle. Solo asociarlas con el
tipo de problema que suelen resolver.

| Familia | Uso habitual |
|---|---|
| ResNet / EfficientNet | Clasificación de imágenes |
| YOLO / Faster R-CNN | Detección de objetos |
| U-Net | Segmentación |
| Mask R-CNN | Detección + máscara por objeto |
| SAM | Generación de máscaras a partir de una guía |

En esta clase usaremos dos ejemplos concretos:

- **ResNet18** para clasificación;
- **YOLO nano** para detección.

La arquitectura interna, el entrenamiento y la comparación entre modelos quedan
para el módulo específico de visión.

---
## Experimento B — clasificación con ResNet18

Vamos a darle una imagen a un modelo llamado **ResNet18**.

El modelo fue entrenado con muchas categorías de objetos cotidianos.

El flujo puede resumirse así:

```text
imagen
  ↓
preparación automática
  ↓
ResNet18
  ↓
lista de categorías posibles
  ↓
mostramos las respuestas más fuertes
```

Cada respuesta viene acompañada por un `score`.

Para esta introducción alcanza con interpretarlo así:

> **cuanto más alto es el score, más fuerte fue la preferencia del modelo
> por esa categoría frente a las otras disponibles.**

Eso no significa que el modelo tenga razón. Puede equivocarse o puede ocurrir
que ninguna de sus categorías represente bien nuestra imagen.

In [ ]:
# ------------------------------------------------------------
# EXPERIMENTO B: CLASIFICACIÓN
# ------------------------------------------------------------

# 1) Cargamos la versión preentrenada de ResNet18.
#    Los "pesos" representan lo que el modelo aprendió previamente.
pesos_resnet = ResNet18_Weights.DEFAULT

# 2) Torchvision también nos da la preparación correcta de la imagen.
preprocesar_resnet = pesos_resnet.transforms()

# 3) Guardamos los nombres de las categorías que conoce el modelo.
categorias_imagenet = pesos_resnet.meta["categories"]

try:
    # Intentamos cargar el modelo YA ENTRENADO.
    clasificador = resnet18(weights=pesos_resnet)
    resnet_preentrenada = True

except Exception as error:
    # Si no hay conexión y los pesos no están guardados,
    # usamos la arquitectura vacía para que el notebook continúe.
    # En ese caso las predicciones NO tienen significado.
    print("AVISO: no se pudieron cargar los pesos preentrenados.")
    print("Se usará la arquitectura sin entrenar:", error)
    clasificador = resnet18(weights=None)
    resnet_preentrenada = False

# Ponemos el modelo en modo de uso/inferencia.
clasificador.eval()


def predecir_top_k(imagen, k=5):
    """
    Recibe una imagen y devuelve las k categorías con mayor score.

    Flujo:
        imagen -> preparar -> modelo -> scores -> ordenar
    """

    # Convertimos la imagen al formato que ResNet18 espera.
    lote = preprocesar_resnet(imagen).unsqueeze(0)

    # Solo queremos obtener una predicción, no entrenar.
    with torch.inference_mode():
        salida = clasificador(lote)
        probabilidades = salida.softmax(dim=1)[0]

    # Elegimos las k respuestas más altas.
    valores, indices = probabilidades.topk(k)

    return pd.DataFrame({
        "clase": [categorias_imagenet[i] for i in indices.tolist()],
        "score": valores.cpu().tolist(),
    })


# Elegimos la imagen que queremos clasificar.
nombre_clasificacion = "flor"
imagen_clasificacion = imagenes[nombre_clasificacion]

# Pedimos las 5 respuestas más fuertes.
ranking_resnet = predecir_top_k(imagen_clasificacion, k=5)

print("Imagen:", nombre_clasificacion)
print("Modelo preentrenado disponible:", resnet_preentrenada)

display(ranking_resnet.style.format({"score": "{:.3f}"}))

In [ ]:
# Mostramos juntos:
#   izquierda -> la imagen que recibió el modelo
#   derecha   -> las categorías que propuso

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(imagen_clasificacion)
axes[0].set_title("Lo que recibió el modelo")
axes[0].axis("off")

axes[1].barh(
    ranking_resnet["clase"][::-1],
    ranking_resnet["score"][::-1],
)
axes[1].set_xlim(0, max(1.0, ranking_resnet["score"].max() * 1.1))
axes[1].set_xlabel("score")
axes[1].set_title("Las 5 respuestas más fuertes")

plt.tight_layout()
plt.show()

# Solo como observación: vemos la forma numérica de la imagen
# después de prepararla para ResNet18.
tensor_preprocesado = preprocesar_resnet(imagen_clasificacion)
print("Forma de la imagen preparada:", tuple(tensor_preprocesado.shape))

### Cómo interpretar el resultado

No preguntes solamente:

> “¿Cuál tuvo el score más alto?”

Preguntá también:

1. ¿La primera categoría describe razonablemente la imagen?
2. ¿Las siguientes alternativas tienen sentido?
3. ¿Hay alguna categoría claramente absurda?
4. ¿El modelo conoce realmente el tipo de objeto que queremos reconocer?

Un score alto **no prueba** que la respuesta sea correcta.

Para saber si un modelo funciona de verdad necesitamos evaluarlo con muchas
imágenes cuya respuesta correcta ya conocemos. En este experimento solo estamos
observando su comportamiento.

---
## Experimento C — detección con YOLO

Ahora cambia la pregunta.

Ya no queremos solamente saber **qué hay**. Queremos saber:

> **¿qué objetos hay y dónde están?**

Usaremos un modelo YOLO pequeño, preentrenado para reconocer objetos cotidianos.

La salida de un detector puede imaginarse así:

```text
imagen
  ↓
YOLO
  ↓
persona   + rectángulo + score
auto      + rectángulo + score
colectivo + rectángulo + score
...
```

Cada rectángulo se llama **bounding box** o caja.

### El umbral

El modelo puede producir candidatos débiles y fuertes. Nosotros elegimos un
`UMBRAL_VISUAL`.

- score menor al umbral -> no lo mostramos;
- score igual o mayor -> lo mostramos.

Por eso cambiar el umbral puede cambiar el número de objetos visibles.

Si YOLO no puede descargarse o ejecutarse, el notebook seguirá funcionando y
mostrará un aviso.

In [ ]:
# ------------------------------------------------------------
# EXPERIMENTO C: DETECCIÓN
# ------------------------------------------------------------

# Elegimos una escena donde tenga sentido buscar varios objetos.
imagen_deteccion_nombre = (
    "deteccion_calle"
    if "deteccion_calle" in imagenes
    else "paisaje"
)
imagen_deteccion = imagenes[imagen_deteccion_nombre]

# Preparamos contenedores vacíos.
# Si YOLO falla, el notebook puede continuar sin inventar resultados.
cajas_yolo = np.empty((0, 4), dtype=float)
scores_yolo = np.empty(0, dtype=float)
clases_yolo = np.empty(0, dtype=int)
nombres_yolo = {}
yolo_disponible = False
error_yolo = None

try:
    from ultralytics import YOLO

    # Carpeta donde se puede guardar el modelo descargado.
    cache_modelos = Path("modelos_cache")
    cache_modelos.mkdir(exist_ok=True)
    ruta_yolo = cache_modelos / "yolo11n.pt"

    # Cargamos YOLO nano.
    detector_yolo = YOLO(str(ruta_yolo))

    # Ejecutamos el detector sobre la imagen.
    # conf=0.10 conserva candidatos desde score 0.10.
    resultado_yolo = detector_yolo.predict(
        source=imagen_deteccion,
        imgsz=480 if FAST_MODE else 640,
        conf=0.10,
        device="cpu",
        verbose=False,
    )[0]

    # Extraemos:
    #   caja  -> dónde está el objeto
    #   score -> fuerza de la predicción
    #   clase -> qué objeto cree que es
    cajas_yolo = resultado_yolo.boxes.xyxy.cpu().numpy()
    scores_yolo = resultado_yolo.boxes.conf.cpu().numpy()
    clases_yolo = resultado_yolo.boxes.cls.cpu().numpy().astype(int)
    nombres_yolo = resultado_yolo.names

    yolo_disponible = True

except Exception as error:
    error_yolo = f"{type(error).__name__}: {error}"


print("Imagen:", imagen_deteccion_nombre)
print("YOLO ejecutado:", yolo_disponible)
print("Candidatos encontrados desde score 0.10:", len(cajas_yolo))

if error_yolo:
    print("YOLO no estuvo disponible:", error_yolo)

In [ ]:
# ------------------------------------------------------------
# VARIABLE PARA EXPERIMENTAR
# ------------------------------------------------------------
# Probá luego con:
#   0.10 -> mostrar más candidatos
#   0.25 -> valor intermedio
#   0.50 -> exigir una respuesta más fuerte
UMBRAL_VISUAL = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(imagen_deteccion)

mostradas = 0

for caja, score, clase in zip(cajas_yolo, scores_yolo, clases_yolo):

    # Si el score es menor al umbral, no mostramos esa detección.
    if score < UMBRAL_VISUAL:
        continue

    # La caja tiene cuatro coordenadas:
    # izquierda, arriba, derecha, abajo.
    x1, y1, x2, y2 = caja

    # Dibujamos el rectángulo.
    ax.add_patch(Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2.5,
    ))

    # Convertimos el número de clase en un nombre legible.
    etiqueta = nombres_yolo.get(int(clase), str(clase))

    # Escribimos nombre + score.
    ax.text(
        x1,
        max(0, y1 - 4),
        f"{etiqueta}: {score:.2f}",
        fontsize=10,
        backgroundcolor="white",
    )

    mostradas += 1

if yolo_disponible:
    titulo = f"{mostradas} detección(es) con score >= {UMBRAL_VISUAL}"
else:
    titulo = "YOLO no disponible: no se muestran detecciones"

ax.set_title(titulo)
ax.axis("off")
plt.tight_layout()
plt.show()

### Qué aprendemos moviendo el umbral

El umbral es una **decisión nuestra**, no del modelo.

En general:

- un umbral bajo muestra más resultados, incluidos algunos dudosos;
- un umbral alto muestra menos resultados y exige mayor score.

```text
más permisivo  <-------------------->  más exigente
más candidatos                         menos candidatos
```

En un sistema real elegiríamos el umbral según el problema.

Si es muy costoso **no detectar** una falla, podríamos preferir un sistema más
sensible aunque produzca más falsas alarmas.

El punto clave es que la salida del modelo todavía necesita criterios de uso.

---
## 5. ¿Cómo sabemos si un modelo funciona bien?

Mirar una o dos imágenes sirve para aprender, pero no alcanza para evaluar un sistema.

Necesitamos imágenes donde conocemos la respuesta correcta y comparar.

| Caso | Significado |
|---|---|
| Verdadero positivo | Había algo y el modelo lo encontró |
| Falso positivo | El modelo dijo que había algo, pero no |
| Falso negativo | Había algo y el modelo no lo encontró |
| Verdadero negativo | No había nada y el modelo lo descartó |

De estos casos surgen métricas conocidas:

- **accuracy:** proporción total de aciertos;
- **precision:** cuando el modelo dijo “positivo”, cuántas veces acertó;
- **recall:** de los positivos reales, cuántos encontró;
- **F1:** una forma de balancear precision y recall.

No hace falta memorizar las fórmulas en esta introducción.
Lo importante es entender que **cada métrica responde una pregunta distinta**.

Ejemplo: si una falla aparece solamente en 1 de cada 100 imágenes, un modelo que
siempre diga “sin falla” tendría 99 % de accuracy y aun así sería inútil para
encontrar fallas.

In [ ]:
# Ejemplo pequeño y controlado.
#
# 1 = caso positivo
# 0 = caso negativo
#
# y_real -> lo que realmente ocurrió
# y_pred -> lo que predijo el modelo

y_real = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])

metricas_clasificacion = pd.Series({
    "accuracy": accuracy_score(y_real, y_pred),
    "precision": precision_score(y_real, y_pred),
    "recall": recall_score(y_real, y_pred),
    "F1": f1_score(y_real, y_pred),
})

metricas_clasificacion.round(3).to_frame("valor")

### IoU — ¿qué tan bien quedó ubicada una caja?

En detección no alcanza con acertar la categoría.

Si el modelo dice “persona” pero dibuja el rectángulo muy lejos de la persona
real, la localización es mala.

Para comparar dos cajas se usa con frecuencia una medida llamada **IoU
(Intersection over Union)**.

La intuición es:

```text
IoU = cuánto se superponen / cuánto ocupan entre las dos
```

Sus valores van aproximadamente de:

- `0` -> las cajas no coinciden;
- `1` -> las cajas coinciden perfectamente.

No necesitamos calcularlo a mano. Vamos a verlo dibujando una caja real y una
caja predicha.

In [ ]:
def validar_caja(caja):
    """Comprueba que la caja tenga ancho y alto mayores que cero."""
    x1, y1, x2, y2 = np.asarray(caja, dtype=float)

    if x2 <= x1 or y2 <= y1:
        raise ValueError("La caja debe cumplir x2 > x1 e y2 > y1.")

    return np.array([x1, y1, x2, y2], dtype=float)


def iou_cajas(caja_a, caja_b):
    """
    Calcula cuánto se superponen dos rectángulos.

    Regla intuitiva:
        misma posición    -> IoU cerca de 1
        poca coincidencia -> IoU cerca de 0
    """
    a = validar_caja(caja_a)
    b = validar_caja(caja_b)

    # Zona compartida por las dos cajas.
    inter_x1 = max(a[0], b[0])
    inter_y1 = max(a[1], b[1])
    inter_x2 = min(a[2], b[2])
    inter_y2 = min(a[3], b[3])

    ancho_inter = max(0.0, inter_x2 - inter_x1)
    alto_inter = max(0.0, inter_y2 - inter_y1)
    interseccion = ancho_inter * alto_inter

    # Área de cada caja.
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])

    # Todo lo cubierto por ambas cajas.
    union = area_a + area_b - interseccion

    return interseccion / union


# Esta es la caja que tomamos como correcta.
caja_real = np.array([1.0, 1.0, 7.0, 6.0])

# Esta representa una predicción del modelo.
caja_predicha = np.array([2.0, 0.5, 8.0, 5.0])

iou_box = iou_cajas(caja_real, caja_predicha)

# Dibujamos ambas para interpretar visualmente el número.
fig, ax = plt.subplots(figsize=(6, 5))

for caja, etiqueta in [
    (caja_real, "caja correcta"),
    (caja_predicha, "caja predicha"),
]:
    x1, y1, x2, y2 = caja

    ax.add_patch(Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=3,
        label=etiqueta,
    ))

ax.set(xlim=(0, 9), ylim=(7, 0), aspect="equal")
ax.grid(alpha=0.25)
ax.legend()
ax.set_title(f"IoU = {iou_box:.3f}")
plt.show()

### IoU también puede aplicarse a máscaras

En segmentación trabajamos con regiones más precisas que una caja.

Una **máscara** marca qué píxeles pertenecen al objeto.

Podemos comparar:

```text
máscara correcta
       vs.
máscara predicha
```

La lógica de IoU es la misma:

- mucha superposición -> valor alto;
- poca superposición -> valor bajo.

El próximo ejemplo usa formas artificiales para que el concepto se vea
claramente. No estamos ejecutando un modelo de segmentación; estamos aprendiendo
a interpretar su posible salida.

In [ ]:
def iou_mascaras(mascara_a, mascara_b):
    """Calcula la superposición entre dos máscaras binarias."""
    a = np.asarray(mascara_a, dtype=bool)
    b = np.asarray(mascara_b, dtype=bool)

    if a.shape != b.shape:
        raise ValueError("Las máscaras deben tener la misma forma.")

    interseccion = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    return float(interseccion / union) if union else 1.0


# Creamos dos regiones ovaladas:
# una representa la respuesta correcta y otra la predicción.
yy, xx = np.ogrid[:80, :100]

mascara_real = (
    ((xx - 43) / 26) ** 2
    + ((yy - 40) / 23) ** 2
    <= 1
)

mascara_predicha = (
    ((xx - 50) / 25) ** 2
    + ((yy - 37) / 20) ** 2
    <= 1
)

# Imagen auxiliar para ver dónde se superponen.
superposicion = (
    mascara_real.astype(int)
    + 2 * mascara_predicha.astype(int)
)

iou_mask = iou_mascaras(mascara_real, mascara_predicha)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, matriz, titulo in zip(
    axes,
    [mascara_real, mascara_predicha, superposicion],
    [
        "Máscara correcta",
        "Máscara predicha",
        f"Superposición — IoU {iou_mask:.3f}",
    ],
):
    ax.imshow(matriz)
    ax.set_title(titulo)
    ax.axis("off")

plt.tight_layout()
plt.show()

### ¿Y mAP?

Cuando se evalúa un detector completo aparecen métricas como **AP** y **mAP**.

No vamos a calcularlas en esta introducción.

Solo necesitamos quedarnos con la idea:

> mAP resume qué tan bien detecta y localiza objetos un modelo sobre un conjunto
> de imágenes.

IoU participa en esa evaluación porque ayuda a decidir si una caja predicha está
suficientemente cerca de la caja correcta.

Por ahora alcanza con esta cadena conceptual:

```text
modelo detecta objetos
        ↓
comparamos con respuestas correctas
        ↓
medimos clases + ubicación
        ↓
obtenemos métricas
```

El cálculo formal de AP/mAP queda para el módulo de visión.

---
## Actividad guiada — cambiar una decisión y observar el efecto

Antes del experimento final, hacé una modificación simple.

En la celda siguiente hay tres variables:

- `ROTACION_ACTIVIDAD`
- `CAJA_PRED_ACTIVIDAD`
- `UMBRAL_ACTIVIDAD`

Cambialas de a **una por vez**.

La meta no es obtener “el mejor número”, sino explicar la relación entre la
modificación y el resultado.

Probá, por ejemplo:

- rotación de `45`;
- mover la caja predicha hacia la derecha;
- umbral de detección `0.10` y luego `0.50`.

In [ ]:
# ------------------------------------------------------------
# CAMBIÁ ESTAS TRES VARIABLES
# ------------------------------------------------------------

# 1) ¿Cuánto giramos la imagen?
ROTACION_ACTIVIDAD = 10

# 2) ¿Dónde colocamos nuestra "caja predicha" de ejemplo?
CAJA_PRED_ACTIVIDAD = np.array([1.5, 1.0, 7.5, 5.5])

# 3) ¿Qué score mínimo exigimos a las detecciones YOLO?
UMBRAL_ACTIVIDAD = 0.25


# A) Aplicamos la rotación.
imagen_rotada = TF.rotate(imagen_base, ROTACION_ACTIVIDAD)

# B) Comparamos la caja correcta con la caja modificada.
iou_actividad = iou_cajas(caja_real, CAJA_PRED_ACTIVIDAD)

# C) Contamos cuántas detecciones YOLO superan el umbral elegido.
detecciones_actividad = int(
    np.sum(scores_yolo >= UMBRAL_ACTIVIDAD)
)


fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(imagen_rotada)
axes[0].set_title(f"Rotación: {ROTACION_ACTIVIDAD}°")
axes[0].axis("off")

axes[1].bar(
    ["IoU de caja", "detecciones visibles"],
    [iou_actividad, detecciones_actividad],
)
axes[1].set_title(f"Umbral YOLO: {UMBRAL_ACTIVIDAD:.2f}")
axes[1].set_ylim(0, max(1.0, detecciones_actividad + 0.5))

plt.tight_layout()
plt.show()


pd.Series({
    "IoU de la caja": iou_actividad,
    "detecciones sobre el umbral": detecciones_actividad,
    "YOLO disponible": yolo_disponible,
}).to_frame("resultado")

---
# Experimento final — ahora vos manejás el modelo

Este es el experimento de cierre.

La práctica reproduce el flujo mental que usaríamos frente a cualquier sistema
de visión:

```text
1. elijo una imagen
2. se la doy al modelo
3. observo qué responde
4. modifico una decisión
5. comparo el resultado
6. explico qué cambió
```

## Tu tarea

En la próxima celda modificá solamente estas variables:

```python
IMAGEN_PRACTICA = "paisaje"
TOP_K_PRACTICA = 3
UMBRAL_PRACTICA = 0.25
```

También podés usar `"flor"` o `"deteccion_calle"` si está disponible.

El notebook ejecutará dos tareas sobre la misma imagen:

- **clasificación:** ¿qué categorías propone ResNet18?
- **detección:** ¿qué objetos muestra YOLO con el umbral elegido?

Después hacé al menos dos ejecuciones cambiando la imagen o el umbral.

### Preguntas para responder

1. ¿Qué recibió cada modelo?
2. ¿Qué devolvió ResNet18?
3. ¿Qué devolvió YOLO?
4. ¿Qué ocurrió cuando cambiaste el umbral?
5. ¿Hubo alguna predicción que no te resultó razonable?
6. ¿Qué diferencia observás entre **resultado del modelo** y **decisión del usuario**?

La práctica se completa cuando podés explicar el flujo con tus propias palabras.

In [ ]:
# ============================================================
# EXPERIMENTO FINAL
# ============================================================
# MODIFICÁ SOLO ESTAS VARIABLES:

IMAGEN_PRACTICA = "paisaje"
TOP_K_PRACTICA = 3
UMBRAL_PRACTICA = 0.25


# ------------------------------------------------------------
# 1. Elegimos la imagen
# ------------------------------------------------------------
if IMAGEN_PRACTICA not in imagenes:
    raise ValueError(
        f"Imagen no disponible. Elegí una de: {list(imagenes)}"
    )

imagen_practica = imagenes[IMAGEN_PRACTICA]


# ------------------------------------------------------------
# 2. CLASIFICACIÓN
# ------------------------------------------------------------
# ResNet18 recibe toda la imagen y devuelve categorías.
ranking_practica = predecir_top_k(
    imagen_practica,
    k=TOP_K_PRACTICA,
)


# ------------------------------------------------------------
# 3. DETECCIÓN
# ------------------------------------------------------------
# Ejecutamos YOLO sobre ESA MISMA imagen.
cajas_practica = np.empty((0, 4), dtype=float)
scores_practica = np.empty(0, dtype=float)
clases_practica = np.empty(0, dtype=int)
nombres_practica = {}
deteccion_ok = False

if yolo_disponible:
    resultado_practica = detector_yolo.predict(
        source=imagen_practica,
        imgsz=480 if FAST_MODE else 640,
        conf=0.10,
        device="cpu",
        verbose=False,
    )[0]

    cajas_practica = resultado_practica.boxes.xyxy.cpu().numpy()
    scores_practica = resultado_practica.boxes.conf.cpu().numpy()
    clases_practica = (
        resultado_practica.boxes.cls.cpu().numpy().astype(int)
    )
    nombres_practica = resultado_practica.names
    deteccion_ok = True


# ------------------------------------------------------------
# 4. MOSTRAMOS LOS DOS TIPOS DE RESPUESTA
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: imagen + cajas de YOLO
axes[0].imshow(imagen_practica)
cantidad_visible = 0

for caja, score, clase in zip(
    cajas_practica,
    scores_practica,
    clases_practica,
):
    if score < UMBRAL_PRACTICA:
        continue

    x1, y1, x2, y2 = caja

    axes[0].add_patch(Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2.5,
    ))

    nombre = nombres_practica.get(int(clase), str(clase))

    axes[0].text(
        x1,
        max(0, y1 - 4),
        f"{nombre}: {score:.2f}",
        fontsize=9,
        backgroundcolor="white",
    )

    cantidad_visible += 1

axes[0].set_title(
    f"YOLO — {cantidad_visible} detección(es), "
    f"umbral {UMBRAL_PRACTICA}"
)
axes[0].axis("off")


# Derecha: ranking de ResNet18
axes[1].barh(
    ranking_practica["clase"][::-1],
    ranking_practica["score"][::-1],
)
axes[1].set_title(f"ResNet18 — top {TOP_K_PRACTICA}")
axes[1].set_xlabel("score")

plt.suptitle(f"Imagen de práctica: {IMAGEN_PRACTICA}", fontsize=14)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. RESUMEN
# ------------------------------------------------------------
print("Imagen usada:", IMAGEN_PRACTICA)
print("Detección YOLO disponible:", deteccion_ok)
print("Detecciones visibles:", cantidad_visible)

display(ranking_practica.style.format({"score": "{:.3f}"}))

---
## Hasta dónde llega esta introducción

En esta clase usamos modelos ya entrenados y ejemplos pequeños para comprender
su funcionamiento general.

Eso **no permite concluir** que el mismo modelo funcionará bien en un problema real.

Para construir una solución de visión aplicada todavía haría falta:

- definir con precisión qué queremos detectar o clasificar;
- reunir imágenes representativas;
- disponer de respuestas correctas para evaluar;
- adaptar o entrenar un modelo cuando sea necesario;
- elegir métricas adecuadas;
- probarlo con imágenes que el modelo no haya visto.

Esos temas pertenecen a una etapa posterior.

La idea que sí deberías llevarte de esta clase es:

```text
problema
   ↓
tipo de tarea
   ↓
imagen
   ↓
modelo
   ↓
salida
   ↓
interpretación y evaluación
```

## Síntesis

- **Clasificación** responde qué categoría representa una imagen.
- **Detección** agrega dónde se encuentran los objetos.
- **Segmentación** marca regiones píxel por píxel.
- Un modelo preentrenado permite experimentar sin entrenar desde cero.
- La imagen debe prepararse en el formato esperado por el modelo.
- Un `score` alto no garantiza que una predicción sea correcta.
- Un `threshold` es una decisión que cambia qué resultados aceptamos.
- **IoU** ayuda a medir cuánto coincide una región predicha con la correcta.
- Evaluar un modelo requiere muchas imágenes con respuestas conocidas.

## Comprobación final

Si podés responder estas cuatro preguntas con un ejemplo del notebook,
alcanzaste el objetivo introductorio:

1. ¿Qué entra al modelo?
2. ¿Qué tipo de salida produce?
3. ¿Qué decisión podemos modificar nosotros?
4. ¿Por qué mirar una sola predicción no alcanza para afirmar que el modelo funciona bien?

En el módulo específico de visión se profundizarán entrenamiento,
transfer learning, datasets, arquitecturas, métricas y evaluación.